# cAPTure: corrected graph-logit development comparison

This notebook evaluates the four completed graph models using their immutable rescored raw logits. It repeats the frozen development-OOF alert-budget analysis and places the results beside the four previously validated XGBoost development benchmarks.

XGBoost is an external benchmark, not an architecture-matched ablation. No model is trained, no checkpoint is loaded, and no held-out scenario is accessed.

## 1. Mount Drive and prepare the CPU environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import Markdown, display
from utils.capture_graph_logit_comparison import (
    run_graph_logit_comparison,
    validate_graph_logit_comparison,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("A GPU runtime is not required for this comparison.")

## 2. Bind the immutable graph and XGBoost evidence

In [ ]:
RESCORING_RUN_ID = "20261001_graph_oof_logit_rescoring_v1"
SOURCE_GRAPH_COMPARISON_RUN_ID = (
    "20261001T153351_608797Z_graph_four_model_comparison"
)
SATURATION_AUDIT_RUN_ID = (
    "20261001T175306_830261Z_graph_score_saturation_audit"
)
XGB_OPERATIONAL_RUN_ID = (
    "20260923T153258_885662Z_operational_oof_v2_float64"
)
XGB_EARLY_WARNING_RUN_ID = (
    "20260923T155543_771684Z_early_warning_oof_v2_float64"
)
COMPARISON_RUN_ID = "20261001_graph_logit_comparison_v1"

RESCORING_ROOT = (
    DRIVE_ROOT / "graph_oof_rescoring_runs" / RESCORING_RUN_ID
)
SOURCE_GRAPH_COMPARISON_DIR = (
    DRIVE_ROOT / "graph_comparison_runs" / SOURCE_GRAPH_COMPARISON_RUN_ID
)
SATURATION_AUDIT_DIR = (
    DRIVE_ROOT / "graph_score_saturation_audit_runs"
    / SATURATION_AUDIT_RUN_ID
)
XGB_OPERATIONAL_DIR = (
    DRIVE_ROOT / "operational_oof_runs" / XGB_OPERATIONAL_RUN_ID
)
XGB_EARLY_WARNING_DIR = (
    DRIVE_ROOT / "early_warning_oof_runs" / XGB_EARLY_WARNING_RUN_ID
)
XGB_ABLATION_DIR = (
    DRIVE_ROOT / "xgb_p_t_ablation_runs"
    / "20260920T000745_916321Z_xgb_p_t_ablation"
)
XGB_RUN_DIRS = {
    "xgb_p": DRIVE_ROOT / "xgb_p_runs/20260919T151844_852400Z_xgb_p",
    "current_window": XGB_ABLATION_DIR / "current_window",
    "history": XGB_ABLATION_DIR / "history",
    "full": DRIVE_ROOT / "xgb_p_t_runs/20260919T201909_754628Z_xgb_p_t",
}
OUTPUT_DIR = (
    DRIVE_ROOT / "graph_logit_comparison_runs" / COMPARISON_RUN_ID
)
MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
TRAINING_CONFIG_PATH = PROJECT_ROOT / "configs/capture_graph_training_v1.yaml"
TERMINAL_POLICY_PATH = (
    PROJECT_ROOT / "configs/capture_early_warning_audit_v1.yaml"
)
RESCORING_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_oof_rescoring_v1.yaml"
)
COMPARISON_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_logit_comparison_v1.yaml"
)

required_paths = [
    RESCORING_ROOT, SOURCE_GRAPH_COMPARISON_DIR, SATURATION_AUDIT_DIR,
    XGB_OPERATIONAL_DIR, XGB_EARLY_WARNING_DIR, MANIFEST_PATH,
    TRAINING_CONFIG_PATH, TERMINAL_POLICY_PATH, RESCORING_CONFIG_PATH,
    COMPARISON_CONFIG_PATH, *XGB_RUN_DIRS.values(),
]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")
print("Graph rescoring input:", RESCORING_ROOT)
print("XGB operational input:", XGB_OPERATIONAL_DIR)
print("Corrected comparison output:", OUTPUT_DIR)

## 3. Run or validate the corrected comparison

The evaluation reads large OOF Parquet files and can take several minutes on CPU. Set `RUN_COMPARISON=True` only after reviewing every bound path. A completed output is immutable and is checksum-validated on a rerun.

In [ ]:
RUN_COMPARISON = False

comparison_arguments = {
    "comparison_config_path": COMPARISON_CONFIG_PATH,
    "rescoring_config_path": RESCORING_CONFIG_PATH,
    "rescoring_root": RESCORING_ROOT,
    "source_comparison_dir": SOURCE_GRAPH_COMPARISON_DIR,
    "saturation_audit_dir": SATURATION_AUDIT_DIR,
    "manifest_path": MANIFEST_PATH,
    "training_config_path": TRAINING_CONFIG_PATH,
    "terminal_policy_path": TERMINAL_POLICY_PATH,
    "xgb_operational_dir": XGB_OPERATIONAL_DIR,
    "xgb_early_warning_dir": XGB_EARLY_WARNING_DIR,
    "xgb_run_dirs": XGB_RUN_DIRS,
    "output_dir": OUTPUT_DIR,
}
if OUTPUT_DIR.exists():
    comparison_report = validate_graph_logit_comparison(**comparison_arguments)
elif RUN_COMPARISON:
    comparison_report = run_graph_logit_comparison(
        **comparison_arguments, batch_size=250_000
    )
else:
    comparison_report = None
    print("Comparison is disabled. No OOF scores were evaluated.")

## 4. Review the corrected metrics

In [ ]:
if comparison_report is None:
    print("Run section 3 with RUN_COMPARISON=True before reviewing results.")
else:
    tables = comparison_report["tables"]
    print("Comparison status:", comparison_report["status"])
    print("Graph operational score field:",
          comparison_report["operational_graph_score_field"])
    print("Graph/XGB packet alignment:",
          comparison_report["aligned_graph_xgb_packet_counts"])
    display(pd.DataFrame(tables["combined_ranking_summary"]))
    display(pd.DataFrame(tables["combined_primary_operational_summary"]))
    display(pd.DataFrame(tables["graph_score_correction_impact"]))
    display(pd.DataFrame(tables["combined_budget_sensitivity_summary"]))
    display(pd.DataFrame(tables["pairwise_comparisons"]))
    display(pd.DataFrame(tables["graph_vs_xgb_benchmark_deltas"]))
    display(pd.DataFrame(tables["corrected_graph_resource_summary"]))
    display(Markdown((OUTPUT_DIR / "review.md").read_text(encoding="utf-8")))

## 5. Provenance boundary

In [ ]:
if comparison_report is not None:
    print("Model training performed:",
          comparison_report["model_training_performed"])
    print("Checkpoint loading performed:",
          comparison_report["checkpoint_loading_performed"])
    print("Held-out scenarios accessed:",
          comparison_report["held_out_scenarios_accessed"])
    print("XGB treated as architecture ablation:",
          not comparison_report["xgb_is_external_development_benchmark_not_architecture_ablation"])
    print("Saved report:", OUTPUT_DIR / "comparison_report.json")
    print("Saved review:", OUTPUT_DIR / "review.md")